# 02 · DeepSeek-OCR

**DeepSeek-OCR** — открытая мульти-модальная модель (≈3B параметров) от DeepSeek-AI; поддерживает grounding-промпт и markdown-вывод. Чекпоинт: [`deepseek-ai/DeepSeek-OCR`](https://huggingface.co/deepseek-ai/DeepSeek-OCR).

Ноутбук берёт subset, сохранённый в `data/subset.json` ноутбуком `01_setup_and_dataset.ipynb`, и прогоняет на нём модель. Результаты записываются в `results/<model>/predictions.jsonl`.

## 1. Установка

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# === Colab / Kaggle bootstrap =================================================
# В Colab клонируем репозиторий проекта (предполагается, что код выложен
# в GitHub) и переходим в его корень. Для локального запуска просто
# проверьте, что текущая рабочая директория — корень ocr_eval/.
import os, sys, pathlib

# Клонируем только если ещё нет (защита от повторного запуска)
if not pathlib.Path('ocr_eval').exists():
    !git clone https://github.com/AStrateg2509/ocr_eval.git

os.chdir('ocr_eval')
sys.path.insert(0, 'src')
print("CWD =", os.getcwd())


## 2. Конфиг и загрузка модели

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord

cfg = load_config('configs/deepseek_ocr.yaml')
print(gpu_info())
cfg

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_REPO = cfg['model']['hf_repo']
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

## 3. Загружаем общий subset

In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

## 4. Инференс

DeepSeek-OCR использует свой удобный метод `model.infer(...)`.

In [ ]:
import os, time, traceback
from pathlib import Path
from PIL import Image

out_dir = Path(cfg['output']['results_dir'])
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'predictions.jsonl'
done = already_processed_ids(out_path)
print(f'уже обработано: {len(done)} / {len(subset)}')

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done:
            continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists():
            continue
        rec = PredictionRecord(page_id=gt.page_id, model='deepseek_ocr')
        try:
            with Timer('infer') as t:
                # API DeepSeek-OCR (см. README модели):
                # model.infer(tokenizer, prompt, image_file, output_path,
                #             base_size, image_size, crop_mode, save_results, test_compress)
                tmp_out = out_dir / f'{gt.page_id}'
                tmp_out.mkdir(parents=True, exist_ok=True)
                result_md = model.infer(
                    tokenizer,
                    prompt=cfg['inference']['prompt'],
                    image_file=str(img_path),
                    output_path=str(tmp_out),
                    base_size=cfg['inference']['base_size'],
                    image_size=cfg['inference']['image_size'],
                    crop_mode=cfg['inference']['crop_mode'],
                    save_results=False,
                    test_compress=cfg['inference']['test_compress'],
                )
            rec.full_text = result_md if isinstance(result_md, str) else str(result_md)
            rec.raw_output = rec.full_text
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

## 5. Превью одного результата

In [ ]:
from src.utils import read_jsonl
preds = read_jsonl(out_path)
print(len(preds), 'записей')
print(preds[0]['full_text'][:1000])

## 6. Освободить GPU
После выгрузки можно запускать следующую модель в этом же runtime.

In [ ]:
del model; cuda_free(); print(gpu_info())